# Issue #234 — unified data source schema contract (BJ-UNIFIED-DATA-SOURCE-SCHEMA-001)

Frozen source registry, canonical fixture identity, decision-epoch semantics and
physical database schema for every #232 recovery branch.  Audit mode: candidate.

This notebook profiles every existing database (tables, keys, time ranges,
nulls, duplicates, source clocks), freezes the source registry and feasibility
matrix, emits the versioned SQL DDL + pandas/PyArrow schemas + domain dictionary
+ identity crosswalk + migration-free replacement plan, and atomically publishes
the immutable results package.

## Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Notebook may be executed from the repo root or from its own directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists():
    ROOT = ROOT.parents[2]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import json

from source_schema import builder, census, contract, ddl, feasibility
from source_schema import handoff, identity, manifest, plan, publication, sources

PACKAGE = contract.package_path(ROOT)
print("root:", ROOT)
print("package:", PACKAGE)
print("pandas:", pd.__version__)
print("contract:", contract.CONTRACT_VERSION, "/", contract.CONTRACT_VERSION_TAG)

root: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_48936420
package: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_48936420/results/bj234_source_schema_contract_v1
pandas: 3.0.3
contract: BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v1


## Census: profile every existing database read-only

Databases are gitignored raw assets on the producer machine; the census opens
them read-only, measures table/column/identity/timestamp facts and records
existence, size and content hashes.  Missing assets are named explicitly rather
than silently omitted.

In [2]:
census_result = census.run_census(ROOT)
print("databases:", len(census_result.databases))
print("tables:", len(census_result.tables))
print("columns:", len(census_result.columns))
print("identity rows:", len(census_result.identity))
print("timestamp rows:", len(census_result.timestamps))
print("source hash rows:", len(census_result.source_hashes))

databases: 7
tables: 63
columns: 1009
identity rows: 7
timestamp rows: 24
source hash rows: 28


## Database inventory and sizes

In [3]:
display(census_result.databases)

,database_key,path,family,description,exists,size_bytes,sha256
0,beidan_xlsm,data/derived/beidan_xlsm/beidan_xlsm.sqlite,INTERNAL_HISTORICAL,2014-2025 historical identity and 51.5M labell...,True,6872944640,7f62079b18b3c028884da92aa8393f5b198246c9d7c32c...
1,beidan_features,data/beidan_features.db,INTERNAL_HISTORICAL,2014-2025 match identities and European probab...,True,47226880,48d73b868235a78d73ea2587d5a2cd9094a861b605bfcd...
2,beidan_history,data/beidan_history.db,INTERNAL_HISTORICAL,2024-2026 settlement/history rows,True,6758400,867e11452306c29e7b64b97af82f730ab0d96f7519d995...
3,beidan_live,data/beidan_live.db,BEIJING_FIRST_PARTY,2026-05 onward timestamped Beijing SP/European...,True,262406144,6ef5cdfad61c74276ac856c18625e82297ad77f4407d07...
4,beidan_modeling_plus,data/derived/beidan_xlsm/beidan_modeling_plus_...,INTERNAL_HISTORICAL,Derived modeling tables built from the xlsm wo...,True,1087799296,c35a42800a7998ede97c5105371cd78a293a151a9c004a...
5,beidan_nowscore,data/derived/beidan_xlsm/beidan_nowscore_2026....,INTERNAL_HISTORICAL,Derived nowscore table built from the xlsm wor...,True,191561728,3948176d42e7799a986e406a75266fe840eadf977c21d8...
6,500_bjdc,data/derived/500_bjdc/500_bjdc.sqlite,BEIJING_FIRST_PARTY,500.bjdc live SP captures and settlement scrapes,True,17091457024,5a5ebf3c35e3b01dd67c863c20ca28c4fc82260e84cc35...


## Table registry (rows, columns, keys, indexes, identity)

In [4]:
display(census_result.tables)

,database_key,table,row_count,column_count,primary_keys,indexes,identity_unique_rows,identity_duplicate_rows
0,beidan_xlsm,action_candidates,1562152,19,,"idx_action_candidates_hcp,idx_action_candidate...",-1,-1
1,beidan_xlsm,action_definitions,8,4,action_code,,-1,-1
2,beidan_xlsm,exotic_odds,7615491,4,,"idx_exotic_market,idx_exotic_match",-1,-1
3,beidan_xlsm,issue_list,27,1,issue,,-1,-1
4,beidan_xlsm,line_lookup,153,9,row_no,,-1,-1
...,...,...,...,...,...,...,...,...
58,500_bjdc,segment_decision_refresh_runs,32,10,"issue,segment_idx,checkpoint",idx_segment_decision_refresh_issue,-1,-1
59,500_bjdc,segment_decision_windows,12,11,"issue,segment_idx",,-1,-1
60,500_bjdc,segment_recovery_publications,1,9,"issue,parent_segment_idx",,-1,-1
61,500_bjdc,segment_recovery_windows,1,8,"issue,parent_segment_idx",,-1,-1


## Timestamp columns and clock semantics (REQ-DS03)

Every measured timestamp column is graded: TIMESTAMPED_UTC vs naive local vs
label-relative vs schedule-only vs settlement-after-event.  Labels such as
`initial/10h/1h/final` are never accepted as exact observation timestamps.

In [5]:
display(census_result.timestamps)

,database_key,table,column,min_value,max_value,null_count,clock_semantics
0,beidan_xlsm,action_candidates,kickoff_ts,1899-12-30 08:00:00,2025-07-13 03:00:00,20480,SCHEDULE_ONLY
1,beidan_xlsm,match_features,kickoff_ts,1899-12-30 08:00:00,2025-07-13 03:00:00,2560,SCHEDULE_ONLY
2,beidan_xlsm,matches,kickoff_ts,1899-12-30 08:00:00,2025-07-13 03:00:00,2560,SCHEDULE_ONLY
3,beidan_xlsm,matches,ts_10h,2013-12-28 06:49:59,2025-07-12 17:00:00,2566,LABEL_RELATIVE
4,beidan_xlsm,official_500_settlements,end_time,2013-12-28 16:35,2017-12-26 02:50,0,SCHEDULE_ONLY
5,beidan_xlsm,official_500_settlements,fetched_at,2026-06-26 12:10:03,2026-06-26 12:14:27,0,TIMESTAMPED_LOCAL_NO_TZ
6,beidan_xlsm,official_500_settlements,imported_at,2026-06-26T12:14:33+00:00,2026-06-26T12:14:33+00:00,0,TIMESTAMPED_UTC
7,beidan_xlsm,outcome_features,kickoff_ts,1899-12-30 08:00:00,2025-07-13 03:00:00,7680,SCHEDULE_ONLY
8,beidan_features,matches,kickoff,08:00:00,2025-04-15T08:30:00,0,SCHEDULE_ONLY
9,beidan_history,historical_matches,kickoff_ts,2024-01-01 14:00:00,2026-12-31 16:00:00,287,SCHEDULE_ONLY


## Identity uniqueness (REQ-DS02)

Canonical fixture identity is `(issue_code, match_idx)`.  The census measures
legacy duplicate identity rows per table; the canonical schema enforces
`UNIQUE(issue_code, match_idx)` and names every legacy defect instead of
silently dropping rows.

In [6]:
identity_checks = identity.build_identity_checks(census_result.identity)
display(identity_checks)

,database_key,table,row_count,distinct_identity,duplicate_identity_rows,uniqueness_verdict,contract_rule
0,beidan_xlsm,match_features,195269,192709,2560,DUPLICATE_DEFECT,"UNIQUE(issue_code, match_idx)"
1,beidan_xlsm,matches,195269,192709,2560,DUPLICATE_DEFECT,"UNIQUE(issue_code, match_idx)"
2,beidan_xlsm,official_500_settlements,74295,74295,0,UNIQUE,"UNIQUE(issue_code, match_idx)"
3,beidan_xlsm,outcome_features,585807,192709,393098,DUPLICATE_DEFECT,"UNIQUE(issue_code, match_idx)"
4,beidan_history,historical_matches,38463,38463,0,UNIQUE,"UNIQUE(issue_code, match_idx)"
5,beidan_live,matches,5155,5155,0,UNIQUE,"UNIQUE(issue_code, match_idx)"
6,beidan_live,settlements,3349,3349,0,UNIQUE,"UNIQUE(issue_code, match_idx)"


## Source registry and feasibility matrix (REQ-DS01, REQ-DS08)

Every source family is classified by causal admissibility with evidence (clock
semantics + coverage grade + rationale), and each downstream branch consumes a
concrete selected source.

In [7]:
registry = feasibility.source_registry()
display(registry)

matrix = feasibility.feasibility_matrix()
print("selected branches:", sorted(matrix["selected_for_branch"].dropna().unique()))
display(matrix[["family", "source_key", "admissible_use", "clock_semantics",
                "coverage_grade", "selected_for_branch", "rationale"]])

,source_id,provider,endpoint,acquisition_method,terms_status,timezone,clock_semantics,cadence_seconds,parser_version,admissible_use,first_usable_at_utc,last_usable_at_utc,coverage_grade,raw_sha256,notes
0,beijing_official_interface,Beijing Sports Lottery official interface,official 500.com live/issue feed (prospective ...,prospective capture with raw payload + trusted...,"public display, unlicensed redistribution unce...",Asia/Shanghai,TIMESTAMPED_UTC,60,live_collector_1,STRICT_CAUSAL,2026-05-01T00:00:00Z,present,A,,Primary prospective causal source: native hand...
1,beidan_xlsm_workbook,internal collector workbook (beidan.xlsm),data/derived/beidan_xlsm/beidan_xlsm.sqlite,historical workbook conversion (2014-2025),internal research asset,Asia/Shanghai (naive),LABEL_RELATIVE,N/A,xlsm_converter_1,RETROSPECTIVE_PROXY,2014-01-01T00:00:00Z,2025-07-13T00:00:00Z,C,,51.5M labelled snapshots with LABEL_RELATIVE c...
2,beidan_live_db,live collector (2026-05+),data/beidan_live.db:snapshots,prospective timestamped capture,internal research asset,UTC (snapshot_ts) / naive local (created_ts),TIMESTAMPED_UTC,60,live_collector_1,STRICT_CAUSAL,2026-05-02T00:57:41Z,present,A,,Primary prospective causal source; snapshot_ts...
3,odds_api_historical,The Odds API,api.the-odds-api.com (historical snapshots fro...,licensed API purchase,paid license; usage limits,UTC,TIMESTAMPED_UTC,variable (plan-dependent),odds_api_1,STRICT_CAUSAL,2020-01-01T00:00:00Z,present,B,,Selected EU branch source: licensed timestampe...
4,beidan_xlsm_euro_1x2,internal workbook euro_1x2 snapshots,beidan_xlsm.sqlite:market_snapshots (euro_1x2),historical workbook conversion,internal research asset,Asia/Shanghai (naive),LABEL_RELATIVE,N/A,xlsm_converter_1,RETROSPECTIVE_PROXY,2014-01-01T00:00:00Z,2025-07-13T00:00:00Z,C,,EU probability features present historically b...
5,football_data_csv,Football-Data.co.uk,data/raw/football-data/*.csv,historical CSV download,free for research with attribution,naive local (schedule only),SCHEDULE_ONLY,N/A,football_data_csv_1,RETROSPECTIVE_PROXY,2020-01-01T00:00:00Z,2025-07-01T00:00:00Z,C,,Results/open-or-close odds proxy only; cannot ...
6,internal_results_public_football,internal match results + accepted public footb...,results + football-data CSV (causal team/leagu...,internal results + public CSV,internal + attribution,UTC/normalized,TIMESTAMPED_UTC,N/A,power_features_1,STRICT_CAUSAL,2014-01-01T00:00:00Z,present,B,,Causal team/league power features with explici...
7,beidan_features_db,internal features database,data/beidan_features.db:matches,historical build,internal research asset,naive local,SCHEDULE_ONLY,N/A,features_build_1,RETROSPECTIVE_PROXY,2014-01-01T00:00:00Z,2025-04-15T00:00:00Z,C,,190k match identities with EU probabilities; k...
8,beidan_history_db,internal history database,data/beidan_history.db:historical_matches,historical scrape,internal research asset,naive local,SETTLEMENT_AFTER_EVENT,N/A,history_scrape_1,SETTLEMENT_ONLY,2024-01-01T00:00:00Z,2026-12-31T00:00:00Z,D,,Settlement/history rows and average closing od...
9,bj234_canonical_schema,this contract's physical schema,results/bj234_source_schema_contract_v1 (DDL +...,design freeze (Issue #234),frozen contract,UTC,TIMESTAMPED_UTC,N/A,bj234_source_schema_contract_v1,STRICT_CAUSAL,2026-08-16T00:00:00Z,present,A,,The integration branch consumes the canonical ...


selected branches: ['', 'beijing', 'european', 'integration', 'power']


,family,source_key,admissible_use,clock_semantics,coverage_grade,selected_for_branch,rationale
0,BEIJING_FIRST_PARTY,beijing_official_interface,STRICT_CAUSAL,TIMESTAMPED_UTC,A,beijing,Primary prospective causal source: native hand...
1,INTERNAL_HISTORICAL,beidan_xlsm_workbook,RETROSPECTIVE_PROXY,LABEL_RELATIVE,C,,51.5M labelled snapshots with LABEL_RELATIVE c...
2,INTERNAL_HISTORICAL,beidan_live_db,STRICT_CAUSAL,TIMESTAMPED_UTC,A,beijing,Primary prospective causal source; snapshot_ts...
3,EUROPEAN_BOOKMAKER_FEED,odds_api_historical,STRICT_CAUSAL,TIMESTAMPED_UTC,B,european,Selected EU branch source: licensed timestampe...
4,EUROPEAN_BOOKMAKER_FEED,beidan_xlsm_euro_1x2,RETROSPECTIVE_PROXY,LABEL_RELATIVE,C,,EU probability features present historically b...
5,FOOTBALL_DATA_CSV,football_data_csv,RETROSPECTIVE_PROXY,SCHEDULE_ONLY,C,,Results/open-or-close odds proxy only; cannot ...
6,POWER_FEATURES,internal_results_public_football,STRICT_CAUSAL,TIMESTAMPED_UTC,B,power,Causal team/league power features with explici...
7,INTERNAL_HISTORICAL,beidan_features_db,RETROSPECTIVE_PROXY,SCHEDULE_ONLY,C,,190k match identities with EU probabilities; k...
8,INTERNAL_HISTORICAL,beidan_history_db,SETTLEMENT_ONLY,SETTLEMENT_AFTER_EVENT,D,,Settlement/history rows and average closing od...
9,INTERNAL_HISTORICAL,bj234_canonical_schema,STRICT_CAUSAL,TIMESTAMPED_UTC,A,integration,The integration branch consumes the canonical ...


## Versioned SQL DDL (REQ-DS04/DS05)

The DDL executes in SQLite (WAL, one-writer append-only).  Settlement is a
physically separate table; outcome_code domain is exactly 3/1/0; the 0.65
payout factor appears nowhere.

In [8]:
sql_text = ddl.sql_ddl()
print(sql_text[:4000])
print("...")
print("DDL length:", len(sql_text))
print("outcome domain check:", "CHECK (outcome_code IN ('3','1','0'))" in sql_text)
print("0.65 absent:", "0.65" not in sql_text)

-- BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v1 (candidate mode)
-- Versioned physical schema frozen by Issue #234 (Engineer, M1, parent #232).
-- SQLite in WAL mode; one-writer append-only capture/control tables.
-- All timestamps are timezone-aware UTC text (ISO-8601 with Z).
-- outcome_code domain is exactly '3' / '1' / '0' after explicit mapping.
-- The Beijing 0.65 payout coefficient is absent from every table here; it
-- exists only in downstream ticket settlement metadata.

PRAGMA journal_mode = WAL;
PRAGMA synchronous = FULL;

-- ---------------------------------------------------------------------------
-- source_registry: every source carries one admissible-use class, clock
-- semantics, coverage grade and raw-content hash receipt (REQ-DS01/DS07).
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS source_registry (
    source_id            TEXT PRIMARY KEY,
    provider             TEXT NOT NULL,
    endpoint             TEXT N

## PyArrow research-snapshot schemas (REQ-DS05/DS06)

Immutable partitioned Parquet for research reads, with inference / proxy /
settlement namespaces physically distinguishable.

In [9]:
schema_contract = ddl.parquet_schema_contract()
print(json.dumps(schema_contract, indent=2, sort_keys=True)[:3000])
print("...")

{
  "ddl_version": "BJ-UNIFIED-DATA-SOURCE-SCHEMA-001-ddl-v1",
  "namespaces": {
    "inference": {
      "decision_epoch_id": "int64",
      "estimator_id": "string",
      "fixture_id": "int64",
      "handicap_line": "double",
      "issue_code": "string",
      "match_idx": "int32",
      "observed_at_utc": "timestamp[ns, tz=UTC]",
      "outcome_code": "string",
      "probability": "double",
      "provenance_hash": "string",
      "target_market": "string",
      "uncertainty": "double"
    },
    "proxy": {
      "fixture_id": "int64",
      "handicap_line": "double",
      "issue_code": "string",
      "market_type": "string",
      "match_idx": "int32",
      "outcome_code": "string",
      "proxy_grade": "string",
      "snapshot_kind": "string",
      "source_id": "string",
      "value": "double"
    },
    "settlement": {
      "fixture_id": "int64",
      "issue_code": "string",
      "match_idx": "int32",
      "result_code": "string",
      "settled_at_utc": "timestamp

## Domain dictionary and identity crosswalk contracts

In [10]:
print(ddl.domain_dictionary()[:1800])
print("---")
print(identity.identity_crosswalk_contract()[:1600])

# Domain Dictionary (BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v1)

## outcome_code

Exactly `3` (home win), `1` (draw), `0` (away win) after explicit mapping.  Any legacy label (e.g. `3/1/0`, `主/和/客`, `W/D/L`) must be mapped by an explicit mapping table; unmapped values are rejected by the schema CHECK constraints (REQ-DS04).

## market_type

- `euro_1x2` — European 1X2 decimal odds
- `sp_1x2` — Beijing SP 1X2
- `asian_hcap` — Asian handicap (native line)

## snapshot_kind

- `initial`, `10h`, `1h`, `final` — workbook labels.  These are **labels**, not exact observation timestamps (REQ-DS03); a registry row using them carries clock_semantics `LABEL_RELATIVE` unless independently proven.

## admissible_use (REQ-DS01)

| class | meaning |
|---|---|
| STRICT_CAUSAL | observed before decision epoch with trusted clock |
| RETROSPECTIVE_PROXY | timestamp semantics insufficient; proxy only |
| SETTLEMENT_ONLY | after-event truth; never inference input |
| UNUSABLE | no clock / no identity / corrup

## Migration-free replacement plan (deliverable 4)

Obsolete ambiguous schemas are removed from the accepted interface rather than
wrapped in compatibility layers (repo principle 1).

In [11]:
replacement = plan.replacement_plan()
display(replacement)

,legacy_database,legacy_table,verdict,replacement,rationale
0,beidan_xlsm.sqlite,market_snapshots,REDIRECT,"market_observation + market_quote (long Mx3, s...",Legacy wide Mx3 rows with label snapshots map ...
1,beidan_xlsm.sqlite,market_consensus,REMOVE,computed view over market_quote; never a store...,Aggregated consensus is derivable; storing it ...
2,beidan_xlsm.sqlite,exotic_odds,REMOVE,out of accepted scope; market_type domain is 3...,Exotic (goals/half_full/score) markets are not...
3,beidan_xlsm.sqlite,matches,REDIRECT,fixture + team + team_alias + competition + co...,"Identity columns (issue, match_idx) carry 1 du..."
4,beidan_xlsm.sqlite,action_candidates,REMOVE,derived selection views (not a source table),Selection candidates are downstream derived ar...
5,beidan_xlsm.sqlite,outcome_features,REMOVE,probability_estimate + derived feature views,"Feature-engineered columns (kelly_*, euro_* mo..."
6,beidan_xlsm.sqlite,official_500_settlements,REDIRECT,settlement (physically separate namespace),Settlement rows carry result + winning_sp + fe...
7,beidan_xlsm.sqlite,official_3sp_settlements,REDIRECT,settlement (via raw export import),Official 3SP export is a settlement source; #2...
8,beidan_features.db,matches,REDIRECT,fixture + probability_estimate (EU p_close/p_o...,EU probability features map onto canonical pro...
9,beidan_history.db,historical_matches,REDIRECT,settlement + market_quote (avg closing odds as...,Settlement/history rows and average closing od...


## Build the immutable package

In [12]:
receipt = builder.build(ROOT)
target = builder.write_package(receipt, force=True)
print("BUILT", target)
print("disposition:", receipt.manifest.get("disposition"))
print("databases:", receipt.manifest.get("database_count"))
print("tables:", receipt.manifest.get("table_count"))
print("source hashes:", len(receipt.manifest.get("source_hashes", {})))

BUILT /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_48936420/results/bj234_source_schema_contract_v1
disposition: CONTRACT_FROZEN_CANDIDATE
databases: 7
tables: 63
source hashes: 28


## REQ-DS acceptance evidence

In [13]:
evidence = builder._build_req_ds_evidence(receipt)
display(evidence)

,req,criterion,pass,evidence
0,REQ-DS01,every source classified by causal admissibilit...,True,tables/feasibility_matrix.csv (clock_semantics...
1,REQ-DS02,canonical fixture identity unique within issue...,True,"sql/schema.sql UNIQUE(issue_code, match_idx) +..."
2,REQ-DS03,observed/received/decision/kickoff semantics n...,True,contract/domain_dictionary.md + sql/schema.sql
3,REQ-DS04,normalized tables reject duplicate/invalid Mx3...,True,sql/schema.sql CHECK constraints + unique indexes
4,REQ-DS05,inference/proxy/settlement namespaces physical...,True,schemas/parquet_schema.json namespaces
5,REQ-DS06,reuses pandas/PyArrow/SQLite with measured sca...,True,tables/databases.csv (sizes/rows) + contract/d...
6,REQ-DS07,notebook tables bind machine-readable outputs ...,True,MANIFEST.json source_hashes + notebook semanti...
7,REQ-DS08,handoff names exact inputs/interfaces for Beij...,True,tables/feasibility_matrix.csv selected_for_branch


## Package verification (hash + semantic checks)

In [14]:
hashes = sources.verify_package_hashes(target)
handoff.verify_package(target, ROOT)
print(f"VERIFIED {target}: {len(hashes)} files match SHA256SUMS.json")
print("SEMANTIC CHECKS PASS")

VERIFIED /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_48936420/results/bj234_source_schema_contract_v1: 20 files match SHA256SUMS.json
SEMANTIC CHECKS PASS


## Manifest receipt (REQ-DS07)

In [15]:
import json as json_mod
manifest_payload = json_mod.loads((target / "MANIFEST.json").read_text())
print(json_mod.dumps({k: manifest_payload[k] for k in
    ("contract_version", "audit_mode", "package", "code_hash", "notebook_hash",
     "requirements_hash", "disposition", "selected_sources",
     "payout_factor_forbidden_in_tables")}, indent=2, sort_keys=True))
print("source_hashes:", len(manifest_payload.get("source_hashes", {})))

{
  "audit_mode": "candidate",
  "code_hash": "03a3ec76a7bf54c8fcdafc7c7a69fd8e93c26eaaf8c68a66311d5b32678d3ec7",
  "contract_version": "BJ-UNIFIED-DATA-SOURCE-SCHEMA-001",
  "disposition": "CONTRACT_FROZEN_CANDIDATE",
  "notebook_hash": "1285fa1310b1df6b6ccff2145eec4d67105929af8836364da07feadc4324b508",
  "package": "bj234_source_schema_contract_v1",
  "payout_factor_forbidden_in_tables": true,
  "requirements_hash": "4c2368d0fe36fdd257dd707c4e6d0e6b68ab232e7b6541e9df1e2c2cce292c78",
  "selected_sources": {
    "beijing": "beijing_official_interface",
    "european": "odds_api_historical",
    "integration": "bj234_canonical_schema",
    "power": "internal_results_public_football"
  }
}
source_hashes: 28
